# GPS Data Quality Assessment

Systematic quality analysis of GPS data extracted from 9 batches of Simrad EK80 `.raw` files
covering a ~26-hour survey in the Oslofjord (2024-01-31 / 02-01).

Each batch folder was processed in notebook 08 to produce one CSV per folder with columns:
`timestamp, lat, lon, sentence_type, file`.

**Quality dimensions investigated:**
1. NMEA sentence triplicates (GGA / GLL / RMC redundancy)
2. Temporal regularity (fix rate, gaps)
3. Frozen GPS detection (stuck coordinates)
4. Speed anomalies and outliers
5. Position jitter and effective fix rate
6. Per-folder quality scorecard

In [ ]:
import sys
import gc
import math
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import contextily as cx
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize
from IPython.display import display

sys.path.insert(0, str(Path('../../').resolve()))
from raw_sensor_integration.gps_parser import haversine_m

# ---------------------------------------------------------------------------
# Constants
# ---------------------------------------------------------------------------
NOTEBOOK_DIR = Path('.')
CSV_FILES = sorted(NOTEBOOK_DIR.glob('gps_2*.csv'))
SPEED_KNOTS_MAX = 15        # realistic survey-vessel ceiling
MIN_DT_S = 0.5              # minimum trusted time delta for speed calc
COLORS = plt.cm.tab10.colors

# ---------------------------------------------------------------------------
# Load all CSVs
# ---------------------------------------------------------------------------
folders = {}
labels  = []
for i, f in enumerate(CSV_FILES, 1):
    label = f'F{i}'
    labels.append(label)
    df = pd.read_csv(f, parse_dates=['timestamp'])
    folders[label] = df

# Summary table
rows = []
for label, df in folders.items():
    rows.append({
        'folder': label,
        'rows': len(df),
        'files': df['file'].nunique(),
        'start': df['timestamp'].min(),
        'end':   df['timestamp'].max(),
        'unique_pos': len(df.drop_duplicates(subset=['lat', 'lon'])),
    })
overview = pd.DataFrame(rows).set_index('folder')
display(overview)
print(f'\nTotal rows: {overview["rows"].sum():,}')

In [ ]:
# ---------------------------------------------------------------------------
# Recording timeline (Gantt chart)
# ---------------------------------------------------------------------------
fig, ax = plt.subplots(figsize=(14, 5), dpi=130)

for i, (label, df) in enumerate(folders.items()):
    t0 = df['timestamp'].min()
    t1 = df['timestamp'].max()
    c  = COLORS[i % len(COLORS)]
    ax.barh(i, t1 - t0, left=t0, height=0.6, color=c, alpha=0.85, edgecolor='white')
    dur_h = (t1 - t0).total_seconds() / 3600
    ax.text(t0 + (t1 - t0) / 2, i, f'{label}  ({dur_h:.1f} h)',
            ha='center', va='center', fontsize=9, fontweight='bold', color='white')

ax.set_yticks(range(len(folders)))
ax.set_yticklabels(labels)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
ax.xaxis.set_major_locator(mdates.HourLocator(interval=2))
ax.set_xlabel('Time (UTC)')
ax.set_title('Survey Recording Timeline — 2024-01-31 / 02-01')
ax.grid(axis='x', alpha=0.3)
ax.invert_yaxis()
fig.autofmt_xdate()
plt.tight_layout()
plt.show()
plt.close(fig)
gc.collect()

## 1. NMEA Sentence Triplicates

Each GPS fix cycle produces three NMEA sentences — **GGA**, **GLL**, and **RMC** — with
near-identical timestamps and identical coordinates.  The parser's `drop_duplicates` on
`(timestamp, lat, lon)` collapses some but not all of these, because sub-second
timestamp differences survive.  This inflates row counts by roughly **3x**.

In [ ]:
# ---------------------------------------------------------------------------
# Sentence type distribution per folder (stacked horizontal bar)
# ---------------------------------------------------------------------------
type_counts = {}
for label, df in folders.items():
    type_counts[label] = df['sentence_type'].value_counts()
tc = pd.DataFrame(type_counts).T.fillna(0).astype(int)

fig, ax = plt.subplots(figsize=(12, 6), dpi=130)
tc.plot.barh(stacked=True, ax=ax, color=['#2196F3', '#4CAF50', '#FF9800'],
             edgecolor='white', linewidth=0.5)
ax.set_xlabel('Number of rows')
ax.set_title('NMEA Sentence Type Distribution per Folder')
ax.legend(title='Sentence type', loc='lower right')
ax.grid(axis='x', alpha=0.3)

# annotate ratio
for i, label in enumerate(tc.index):
    total = tc.loc[label].sum()
    ax.text(total + 200, i, f'{total:,}', va='center', fontsize=8, color='grey')

plt.tight_layout()
plt.show()
plt.close(fig)
gc.collect()

In [ ]:
# ---------------------------------------------------------------------------
# Deduplication: round to 1 s, prefer GGA, keep one row per fix cycle
# ---------------------------------------------------------------------------
TYPE_RANK = {'$GPGGA': 0, '$GNGGA': 0, '$INGGA': 0,
             '$GPRMC': 1, '$GNRMC': 1,
             '$GPGLL': 2}

deduped = {}
dedup_rows = []
for label, df in folders.items():
    d = df.copy()
    d['ts_round'] = d['timestamp'].dt.round('1s')
    d['type_rank'] = d['sentence_type'].map(TYPE_RANK).fillna(9)
    d = (d.sort_values(['ts_round', 'type_rank'])
          .drop_duplicates(subset='ts_round')
          .sort_values('timestamp')
          .reset_index(drop=True))
    deduped[label] = d
    dedup_rows.append({
        'folder': label,
        'raw_rows': len(df),
        'deduped_rows': len(d),
        'reduction_%': round(100 * (1 - len(d) / len(df)), 1),
    })

dedup_df = pd.DataFrame(dedup_rows).set_index('folder')
display(dedup_df.style.background_gradient(subset=['reduction_%'], cmap='YlOrRd'))
print(f'Average reduction: {dedup_df["reduction_%"].mean():.1f}%')

## 2. Temporal Quality

After deduplication, the inter-fix time delta should cluster around **1.0 s** (the GPS
receiver's update rate).  Large gaps indicate recording interruptions; very small
deltas suggest surviving triplicates.

In [ ]:
# ---------------------------------------------------------------------------
# Time delta histogram (all folders combined, after dedup)
# ---------------------------------------------------------------------------
all_dt = []
for label, d in deduped.items():
    for _, grp in d.groupby('file', sort=False):
        dt = grp['timestamp'].diff().dt.total_seconds().dropna()
        all_dt.append(dt)
all_dt = pd.concat(all_dt)

fig, ax = plt.subplots(figsize=(12, 6), dpi=130)
bins = np.logspace(-2, 3, 80)
ax.hist(all_dt.clip(lower=0.01), bins=bins, color='steelblue', edgecolor='white',
        linewidth=0.3, alpha=0.85)
ax.set_xscale('log')
ax.set_yscale('log')

for thr, clr, lbl in [(0.5, 'orange', '0.5 s'), (1.0, 'green', '1.0 s'),
                        (5.0, 'red', '5.0 s'), (60.0, 'darkred', '60 s')]:
    ax.axvline(thr, color=clr, linestyle='--', linewidth=1.5, alpha=0.8, label=lbl)

ax.set_xlabel('Time delta between consecutive fixes (seconds)')
ax.set_ylabel('Count')
ax.set_title('Inter-Fix Time Delta Distribution (after dedup, all folders)')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.2)
plt.tight_layout()
plt.show()
plt.close(fig)
gc.collect()

In [ ]:
# ---------------------------------------------------------------------------
# Per-folder temporal stats table
# ---------------------------------------------------------------------------
tstats = []
for label, d in deduped.items():
    dts = []
    for _, grp in d.groupby('file', sort=False):
        dts.append(grp['timestamp'].diff().dt.total_seconds().dropna())
    dt = pd.concat(dts) if dts else pd.Series(dtype=float)
    tstats.append({
        'folder': label,
        'fixes': len(d),
        'median_dt (s)': round(dt.median(), 2),
        'mean_dt (s)': round(dt.mean(), 2),
        'max_dt (s)': round(dt.max(), 1),
        'gaps > 5s': int((dt > 5).sum()),
        'gaps > 60s': int((dt > 60).sum()),
        'fix_rate (Hz)': round(1 / dt.median(), 2) if dt.median() > 0 else 0,
    })

tstats_df = pd.DataFrame(tstats).set_index('folder')
display(tstats_df.style
    .background_gradient(subset=['gaps > 5s', 'gaps > 60s'], cmap='YlOrRd')
    .background_gradient(subset=['fix_rate (Hz)'], cmap='YlGn'))

In [ ]:
# ---------------------------------------------------------------------------
# Time gap scatter — shows WHERE gaps occur on the timeline
# ---------------------------------------------------------------------------
fig, ax = plt.subplots(figsize=(14, 5), dpi=130)

for i, (label, d) in enumerate(deduped.items()):
    for _, grp in d.groupby('file', sort=False):
        g = grp.sort_values('timestamp')
        dt = g['timestamp'].diff().dt.total_seconds()
        mask = dt > 5
        if mask.any():
            ax.scatter(g.loc[mask, 'timestamp'], [i] * mask.sum(),
                       s=dt[mask].clip(upper=300) * 2, c='red', alpha=0.5,
                       edgecolors='darkred', linewidths=0.5, zorder=3)

ax.set_yticks(range(len(labels)))
ax.set_yticklabels(labels)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
ax.xaxis.set_major_locator(mdates.HourLocator(interval=2))
ax.set_xlabel('Time (UTC)')
ax.set_title('Time Gaps > 5 s  (circle size = gap duration)')
ax.grid(True, alpha=0.2)
ax.invert_yaxis()
fig.autofmt_xdate()
plt.tight_layout()
plt.show()
plt.close(fig)
gc.collect()

## 3. Frozen GPS Detection

When a GPS receiver loses satellite lock it may continue reporting the **last known
position** instead of indicating no-fix.  This produces valid-looking coordinates that
are completely wrong — the vessel is moving but the reported position is stuck.

Folder 5 is the worst case: **all 24,839 rows** report the same coordinate
`(59.4008, 10.5578)` for over two hours.

In [ ]:
# ---------------------------------------------------------------------------
# Unique coordinate count per folder (bar chart)
# ---------------------------------------------------------------------------
unique_pos = {label: len(d.drop_duplicates(subset=['lat', 'lon']))
              for label, d in deduped.items()}

fig, ax = plt.subplots(figsize=(12, 6), dpi=130)
bar_colors = ['#d32f2f' if v < 10 else 'steelblue' for v in unique_pos.values()]
bars = ax.bar(unique_pos.keys(), unique_pos.values(), color=bar_colors,
              edgecolor='white', linewidth=0.8)

for bar, v in zip(bars, unique_pos.values()):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 30,
            f'{v:,}', ha='center', va='bottom', fontsize=9,
            fontweight='bold' if v < 10 else 'normal',
            color='red' if v < 10 else 'grey')

ax.set_ylabel('Unique (lat, lon) pairs')
ax.set_xlabel('Folder')
ax.set_title('Spatial Diversity — Unique GPS Positions per Folder')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()
plt.close(fig)
gc.collect()

In [ ]:
# ---------------------------------------------------------------------------
# Consecutive identical position analysis
# ---------------------------------------------------------------------------
frozen_rows = []
for label, d in deduped.items():
    same = ((d['lat'].diff().abs() < 1e-8) & (d['lon'].diff().abs() < 1e-8)).sum()
    frozen_rows.append({
        'folder': label,
        'deduped_rows': len(d),
        'consec_identical': int(same),
        'frozen_%': round(100 * same / max(len(d) - 1, 1), 1),
    })

frozen_df = pd.DataFrame(frozen_rows).set_index('folder')

fig, ax = plt.subplots(figsize=(12, 6), dpi=130)
bar_colors = ['#d32f2f' if r['frozen_%'] > 95 else
              '#ff9800' if r['frozen_%'] > 50 else '#4CAF50'
              for _, r in frozen_df.iterrows()]
ax.bar(frozen_df.index, frozen_df['frozen_%'], color=bar_colors,
       edgecolor='white', linewidth=0.8)

for i, (label, r) in enumerate(frozen_df.iterrows()):
    ax.text(i, r['frozen_%'] + 1, f"{r['frozen_%']:.0f}%",
            ha='center', fontsize=9, fontweight='bold')

ax.set_ylabel('Consecutive identical positions (%)')
ax.set_xlabel('Folder')
ax.set_title('Frozen GPS Indicator — Consecutive Identical Coordinates')
ax.set_ylim(0, 110)
ax.axhline(50, color='orange', linestyle='--', alpha=0.5, label='50% threshold')
ax.axhline(95, color='red', linestyle='--', alpha=0.5, label='95% (likely frozen)')
ax.legend(fontsize=9)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()
plt.close(fig)
gc.collect()

In [ ]:
# ---------------------------------------------------------------------------
# Map: all tracks overlaid, frozen folder as red X
# ---------------------------------------------------------------------------
fig, ax = plt.subplots(figsize=(12, 9), dpi=130)

for i, (label, d) in enumerate(deduped.items()):
    is_frozen = unique_pos[label] < 10
    if is_frozen:
        ax.plot(d['lon'].iloc[0], d['lat'].iloc[0], 'X', color='red',
                markersize=18, markeredgecolor='darkred', markeredgewidth=2,
                zorder=10, label=f'{label} (FROZEN)')
    else:
        c = COLORS[i % len(COLORS)]
        ax.plot(d['lon'], d['lat'], '-', color=c, linewidth=1, alpha=0.7,
                label=label, zorder=3)

cx.add_basemap(ax, crs='EPSG:4326', source=cx.providers.CartoDB.Positron,
               zoom=13, attribution_size=6)

ax.set_xlabel('Longitude (\u00b0E)')
ax.set_ylabel('Latitude (\u00b0N)')
ax.set_title('All GPS Tracks — Frozen Folder Highlighted')
ax.legend(fontsize=8, loc='upper left', framealpha=0.9)
ax.grid(True, alpha=0.2)
plt.tight_layout()
plt.show()
plt.close(fig)
gc.collect()

## 4. Speed Analysis and Anomalies

Speed is derived from `haversine_m()` distance between consecutive fixes divided by
the time delta, converted to **knots** (`m/s \u00d7 3600 / 1852`).  Two guard rails:

- **\u0394t < 0.5 s** \u2192 speed set to NaN (near-zero denominators cause extreme spikes)
- **> 15 knots** flagged as unrealistic for a survey vessel (not removed yet \u2014 we want to visualise them first)

Folder 5 (frozen) is excluded from speed analysis.

In [ ]:
# ---------------------------------------------------------------------------
# Compute speeds on deduped data (exclude frozen folders)
# ---------------------------------------------------------------------------
speed_data = {}
speed_stats = []

for label, d in deduped.items():
    if unique_pos[label] < 10:
        continue  # skip frozen folders

    parts = []
    for _, grp in d.groupby('file', sort=False):
        g = grp.sort_values('timestamp').copy()
        g['dt_s'] = g['timestamp'].diff().dt.total_seconds()
        dists = [float('nan')]
        for j in range(1, len(g)):
            dists.append(haversine_m(
                g['lat'].iloc[j-1], g['lon'].iloc[j-1],
                g['lat'].iloc[j],   g['lon'].iloc[j]))
        g['dist_m'] = dists
        g['speed_kn'] = (g['dist_m'] / g['dt_s']) * (3600 / 1852)
        g.loc[g['dt_s'] < MIN_DT_S, ['dist_m', 'speed_kn']] = float('nan')
        parts.append(g)

    combined = pd.concat(parts).reset_index(drop=True)
    speed_data[label] = combined

    valid = combined['speed_kn'].dropna()
    speed_stats.append({
        'folder': label,
        'fixes': len(combined),
        'valid_speeds': len(valid),
        'median (kn)': round(valid.median(), 2),
        'mean (kn)': round(valid.mean(), 2),
        'max (kn)': round(valid.max(), 1),
        'spikes > 15 kn': int((valid > SPEED_KNOTS_MAX).sum()),
        'spikes > 100 kn': int((valid > 100).sum()),
    })

speed_stats_df = pd.DataFrame(speed_stats).set_index('folder')
display(speed_stats_df.style
    .background_gradient(subset=['spikes > 15 kn', 'spikes > 100 kn'], cmap='YlOrRd')
    .background_gradient(subset=['median (kn)'], cmap='YlGn'))

In [ ]:
# ---------------------------------------------------------------------------
# Speed distribution histogram (log y-axis)
# ---------------------------------------------------------------------------
all_speeds = pd.concat([d['speed_kn'].dropna() for d in speed_data.values()])

fig, ax = plt.subplots(figsize=(12, 6), dpi=130)
bins = np.logspace(-2, 4, 100)
ax.hist(all_speeds.clip(lower=0.01), bins=bins, color='steelblue',
        edgecolor='white', linewidth=0.3, alpha=0.85)
ax.set_xscale('log')
ax.set_yscale('log')

for thr, clr, lbl in [(5, '#4CAF50', '5 kn (slow survey)'),
                        (10, '#2196F3', '10 kn (fast survey)'),
                        (SPEED_KNOTS_MAX, '#FF9800', f'{SPEED_KNOTS_MAX} kn (max realistic)'),
                        (100, '#d32f2f', '100 kn (anomalous)')]:
    ax.axvline(thr, color=clr, linestyle='--', linewidth=1.5, alpha=0.8, label=lbl)

ax.set_xlabel('Speed (knots)')
ax.set_ylabel('Count')
ax.set_title('Speed Distribution — All Active Folders (after dedup)')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.2)
plt.tight_layout()
plt.show()
plt.close(fig)
gc.collect()

In [ ]:
# ---------------------------------------------------------------------------
# Map: track coloured by speed + red circles for spikes
# ---------------------------------------------------------------------------
all_valid = pd.concat([d.dropna(subset=['speed_kn']) for d in speed_data.values()])

fig, ax = plt.subplots(figsize=(12, 9), dpi=130)

norm = Normalize(vmin=0, vmax=all_valid['speed_kn'].quantile(0.95))
good = all_valid[all_valid['speed_kn'] <= SPEED_KNOTS_MAX]
bad  = all_valid[all_valid['speed_kn'] > SPEED_KNOTS_MAX]

ax.scatter(good['lon'], good['lat'], c=good['speed_kn'], cmap='plasma',
           norm=norm, s=3, linewidths=0, zorder=3, alpha=0.7)

if len(bad) > 0:
    ax.scatter(bad['lon'], bad['lat'], s=40, facecolors='none',
               edgecolors='red', linewidths=1.5, zorder=5, alpha=0.8,
               label=f'Speed spikes > {SPEED_KNOTS_MAX} kn (n={len(bad)})')

cx.add_basemap(ax, crs='EPSG:4326', source=cx.providers.CartoDB.Positron,
               zoom=13, attribution_size=6)

cbar = fig.colorbar(ScalarMappable(norm=norm, cmap='plasma'), ax=ax,
                    fraction=0.025, pad=0.01)
cbar.set_label('Speed (knots)', fontsize=9)

ax.set_xlabel('Longitude (\u00b0E)')
ax.set_ylabel('Latitude (\u00b0N)')
ax.set_title('GPS Track Coloured by Speed — Red Circles = Anomalous Spikes')
ax.legend(fontsize=9, loc='upper left', framealpha=0.9)
ax.grid(True, alpha=0.15)
plt.tight_layout()
plt.show()
plt.close(fig)
gc.collect()

In [ ]:
# ---------------------------------------------------------------------------
# Speed spike count per folder (grouped bar chart)
# ---------------------------------------------------------------------------
thresholds = [15, 50, 100]
spike_counts = {}
for label, d in speed_data.items():
    v = d['speed_kn'].dropna()
    spike_counts[label] = {f'> {t} kn': int((v > t).sum()) for t in thresholds}

spike_df = pd.DataFrame(spike_counts).T

fig, ax = plt.subplots(figsize=(12, 6), dpi=130)
spike_df.plot.bar(ax=ax, color=['#FF9800', '#f44336', '#b71c1c'],
                  edgecolor='white', linewidth=0.5)
ax.set_ylabel('Number of speed spikes')
ax.set_xlabel('Folder')
ax.set_title('Speed Spike Count by Severity Threshold')
ax.legend(title='Threshold')
ax.grid(axis='y', alpha=0.3)
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()
plt.close(fig)
gc.collect()

In [ ]:
# ---------------------------------------------------------------------------
# Diagnostic scatter: speed vs time delta (log-log)
# ---------------------------------------------------------------------------
fig, ax = plt.subplots(figsize=(12, 6), dpi=130)

for label, d in speed_data.items():
    valid = d.dropna(subset=['speed_kn', 'dt_s'])
    valid = valid[valid['dt_s'] > 0]
    ax.scatter(valid['dt_s'], valid['speed_kn'], s=4, alpha=0.3,
               label=label, zorder=3)

ax.axhline(SPEED_KNOTS_MAX, color='red', linestyle='--', linewidth=1.5,
           alpha=0.7, label=f'{SPEED_KNOTS_MAX} kn ceiling')
ax.axvline(MIN_DT_S, color='orange', linestyle='--', linewidth=1.5,
           alpha=0.7, label=f'{MIN_DT_S} s min \u0394t')

ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('Time delta (seconds)')
ax.set_ylabel('Speed (knots)')
ax.set_title('Speed vs Time Delta — Spikes Cluster at Low \u0394t')
ax.legend(fontsize=8, loc='upper right')
ax.grid(True, alpha=0.2)
plt.tight_layout()
plt.show()
plt.close(fig)
gc.collect()

## 5. Position Jitter & Effective Fix Rate

Even after deduplication, many consecutive fixes share identical coordinates.
The GPS reports at ~1 Hz but its position solution doesn't always change between
updates.  This section compares the **reported fix rate** (1/median\_dt) with the
**effective fix rate** (positions that actually change per second).

In [ ]:
# ---------------------------------------------------------------------------
# Effective vs reported fix rate (paired bar chart)
# ---------------------------------------------------------------------------
rate_rows = []
for label, d in deduped.items():
    if unique_pos[label] < 10:
        continue
    dur_s = (d['timestamp'].max() - d['timestamp'].min()).total_seconds()
    pos_changed = ((d['lat'].diff().abs() > 1e-8) | (d['lon'].diff().abs() > 1e-8)).sum()
    dts = []
    for _, grp in d.groupby('file', sort=False):
        dts.append(grp['timestamp'].diff().dt.total_seconds().dropna())
    med_dt = pd.concat(dts).median() if dts else 1.0
    rate_rows.append({
        'folder': label,
        'reported_hz': round(1 / med_dt, 2) if med_dt > 0 else 0,
        'effective_hz': round(pos_changed / max(dur_s, 1), 3),
    })

rate_df = pd.DataFrame(rate_rows).set_index('folder')

fig, ax = plt.subplots(figsize=(12, 6), dpi=130)
x = np.arange(len(rate_df))
w = 0.35
ax.bar(x - w/2, rate_df['reported_hz'], w, color='steelblue',
       edgecolor='white', label='Reported fix rate')
ax.bar(x + w/2, rate_df['effective_hz'], w, color='#4CAF50',
       edgecolor='white', label='Effective fix rate (position changed)')

ax.set_xticks(x)
ax.set_xticklabels(rate_df.index)
ax.set_ylabel('Fix rate (Hz)')
ax.set_xlabel('Folder')
ax.set_title('Reported vs Effective GPS Fix Rate')
ax.legend(fontsize=10)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()
plt.close(fig)
gc.collect()

In [ ]:
# ---------------------------------------------------------------------------
# Step distance histogram (non-zero steps only)
# ---------------------------------------------------------------------------
all_dists = pd.concat([d['dist_m'].dropna() for d in speed_data.values()])
nonzero = all_dists[all_dists > 0.01]  # > 1 cm

fig, ax = plt.subplots(figsize=(12, 6), dpi=130)
bins = np.logspace(-2, 3, 80)
ax.hist(nonzero.clip(lower=0.01), bins=bins, color='#4CAF50',
        edgecolor='white', linewidth=0.3, alpha=0.85)
ax.set_xscale('log')
ax.set_yscale('log')

for thr, clr, lbl in [(0.1, 'grey', '0.1 m (GPS noise floor)'),
                        (1.0, 'blue', '1.0 m'),
                        (5.0, 'green', '5.0 m (typical at ~5 kn, 1 Hz)')]:
    ax.axvline(thr, color=clr, linestyle='--', linewidth=1.5, alpha=0.7, label=lbl)

ax.set_xlabel('Step distance (metres)')
ax.set_ylabel('Count')
ax.set_title('Inter-Fix Distance Distribution (non-zero steps only)')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.2)
plt.tight_layout()
plt.show()
plt.close(fig)
gc.collect()

## 6. Per-Folder Quality Scorecard

Each folder receives a quality grade based on multiple metrics:

| Grade | Criteria |
|-------|----------|
| **A** | Clean — few spikes, good spatial diversity |
| **B** | Moderate — some speed spikes (10–50) |
| **C** | Poor — many speed spikes (> 50) |
| **F** | Failed — frozen GPS (< 10 unique positions) |

In [ ]:
# ---------------------------------------------------------------------------
# Quality scorecard table
# ---------------------------------------------------------------------------
score_rows = []
for label, d in deduped.items():
    n_unique = unique_pos[label]
    fz = frozen_df.loc[label, 'frozen_%']
    dedup_n = len(d)
    dur_h = (d['timestamp'].max() - d['timestamp'].min()).total_seconds() / 3600

    if label in speed_data:
        sd = speed_data[label]
        v  = sd['speed_kn'].dropna()
        spikes_15 = int((v > SPEED_KNOTS_MAX).sum())
        max_kn = round(v.max(), 1)
        total_km = round(sd['dist_m'].dropna().sum() / 1000, 2)
        med_kn = round(v.median(), 2)
    else:
        spikes_15 = 0
        max_kn = 0
        total_km = 0
        med_kn = 0

    # Grade
    if n_unique < 10:
        grade = 'F'
    elif spikes_15 > 50:
        grade = 'C'
    elif spikes_15 > 10:
        grade = 'B'
    else:
        grade = 'A'

    score_rows.append({
        'folder': label,
        'deduped_rows': dedup_n,
        'unique_pos': n_unique,
        'frozen_%': fz,
        'duration_h': round(dur_h, 1),
        'median_kn': med_kn,
        'max_kn': max_kn,
        'spikes_>15kn': spikes_15,
        'total_km': total_km,
        'grade': grade,
    })

scorecard = pd.DataFrame(score_rows).set_index('folder')

grade_colors = {'A': '#4CAF50', 'B': '#FF9800', 'C': '#f44336', 'F': '#b71c1c'}

def color_grade(val):
    return f'background-color: {grade_colors.get(val, "white")}; color: white; font-weight: bold'

display(scorecard.style
    .map(color_grade, subset=['grade'])
    .background_gradient(subset=['spikes_>15kn', 'frozen_%'], cmap='YlOrRd')
    .background_gradient(subset=['unique_pos', 'total_km'], cmap='YlGn'))

In [ ]:
# ---------------------------------------------------------------------------
# Quality heatmap
# ---------------------------------------------------------------------------
metrics = ['unique_pos', 'frozen_%', 'spikes_>15kn', 'max_kn', 'total_km', 'median_kn']
hm = scorecard[metrics].copy()

# Normalize: 1 = best quality, 0 = worst
hm_norm = hm.copy()
# Higher is better
for col in ['unique_pos', 'total_km', 'median_kn']:
    r = hm[col].max() - hm[col].min()
    hm_norm[col] = (hm[col] - hm[col].min()) / r if r > 0 else 0.5
# Lower is better (invert)
for col in ['frozen_%', 'spikes_>15kn', 'max_kn']:
    r = hm[col].max() - hm[col].min()
    hm_norm[col] = 1 - (hm[col] - hm[col].min()) / r if r > 0 else 0.5

fig, ax = plt.subplots(figsize=(14, 6), dpi=130)
im = ax.imshow(hm_norm.values, cmap='RdYlGn', aspect='auto', vmin=0, vmax=1)

# Annotate with raw values
for i in range(hm.shape[0]):
    for j in range(hm.shape[1]):
        val = hm.iloc[i, j]
        fmt = f'{val:,.0f}' if val > 10 else f'{val:.1f}'
        text_color = 'white' if hm_norm.iloc[i, j] < 0.3 else 'black'
        ax.text(j, i, fmt, ha='center', va='center', fontsize=9,
                fontweight='bold', color=text_color)

ax.set_xticks(range(len(metrics)))
ax.set_xticklabels(['Unique\npositions', 'Frozen %', 'Spikes\n> 15 kn',
                     'Max speed\n(kn)', 'Total dist\n(km)', 'Median speed\n(kn)'],
                    fontsize=9)
ax.set_yticks(range(len(scorecard)))
ax.set_yticklabels(scorecard.index)

# Grade labels on right side
ax2 = ax.twinx()
ax2.set_ylim(ax.get_ylim())
ax2.set_yticks(range(len(scorecard)))
ax2.set_yticklabels([f"  {g}" for g in scorecard['grade']],
                     fontsize=12, fontweight='bold')
for i, g in enumerate(scorecard['grade']):
    ax2.get_yticklabels()[i].set_color(grade_colors.get(g, 'black'))

cbar = fig.colorbar(im, ax=ax, fraction=0.02, pad=0.08)
cbar.set_label('Quality (green = good, red = bad)', fontsize=9)

ax.set_title('GPS Data Quality Heatmap — Per-Folder Scorecard', fontsize=13)
plt.tight_layout()
plt.show()
plt.close(fig)
gc.collect()

## Conclusions & Recommendations

### Findings

1. **NMEA triplicates**: Each fix cycle produces GGA, GLL, and RMC sentences with near-identical
   timestamps, inflating row counts by ~67%.  Deduplication by rounding to 1-second bins and
   preferring GGA (carries altitude and fix quality) is essential before any analysis.

2. **Frozen GPS (F5)**: Folder 5 (`drive-download-20260317T115045Z`) has **zero spatial information**.
   All 24,839 rows report coordinate `(59.4008, 10.5578)` for 2+ hours (16:52 \u2013 19:10 UTC).
   The GPS receiver lost satellite lock and held the last known fix.  This folder must be
   excluded or flagged as invalid in downstream processing.

3. **Speed spikes**: Caused by near-zero time deltas between consecutive sentences that survived
   deduplication.  After filtering \u0394t < 0.5 s, most extreme spikes disappear.  Remaining
   outliers above 15 knots are GPS coordinate glitches, not vessel motion.

4. **Effective fix rate**: The GPS reports at ~1 Hz but position only changes at ~0.2\u20130.5 Hz
   across most folders.  Between real updates the receiver echoes the previous fix.

### Recommended Cleaning Pipeline

```python
# 1. Deduplicate: round to 1s, prefer GGA
# 2. Exclude frozen folders (unique positions < 10)
# 3. Filter dt < 0.5s (set speed to NaN)
# 4. Flag/remove speed > 15 knots as GPS artifacts
```